In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install -q -U git+https://github.com/huggingface/accelerate.git
!pip install trl
!pip install -q datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_Token")
login(token=hf_token)
print("Hugging Face authentication completed.")

Hugging Face authentication completed.


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto',torch_dtype=torch.float16)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [5]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [6]:
total_param = sum(p.numel() for p in model.parameters())
trainable_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total Model Parameters: ",total_param)
print("Total Model Trainable Parameters: ",trainable_param)

Total Model Parameters:  749275136
Total Model Trainable Parameters:  262735872


In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
non_quantized_model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto')

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [8]:
print(non_quantized_model)
total_param = sum(p.numel() for p in non_quantized_model.parameters())
trainable_param = sum(p.numel() for p in non_quantized_model.parameters() if p.requires_grad)
print("Total Model Parameters: ",total_param)
print("Total Model Trainable Parameters: ",trainable_param)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [9]:
inputs = tokenizer(f"Instruction:\n How does Large Language Models respond to queries? \n\nResponse:\n", return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Instruction:
 How does Large Language Models respond to queries? 

Response:
    Large Language Models can be trained to respond to a wide range of queries. 
    These models can be trained using a variety of methods, including neural networks, recurrent neural networks, and transformer networks. 
    The most common approach is to use a pre-trained language model, such as a GPT-2 or GPT-3 model, and then train the model to respond to a query. 
    This approach works well for large language models, as the model can be trained to respond


In [10]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [11]:
total_param = sum(p.numel() for p in model.parameters())
trainable_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total Model Parameters: ",total_param)
print("Total Model Trainable Parameters: ",trainable_param)

Total Model Parameters:  749275136
Total Model Trainable Parameters:  0


In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [13]:
total_param = sum(p.numel() for p in model.parameters())
trainable_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_param:,}")
print(f"Trainable parameters: {trainable_param:,}")

Total parameters: 754,911,232
Trainable parameters: 5,636,096


In [14]:
from datasets import load_dataset

data = load_dataset("databricks/databricks-dolly-15k", split="train")

tokenizer.pad_token = tokenizer.eos_token

def combine(example):
    if example["context"]:
        prompt = f"Context:\n{example['context']}\n\nInstruction:\n{example['instruction']}\n\nResponse:\n{example['response']}"
    else:
        prompt = f"Instruction:\n{example['instruction']}\n\nResponse:\n{example['response']}"
    return {"text": prompt}

data = data.filter(lambda x: x["response"] is not None and len(x["response"]) > 0)
data = data.map(combine, remove_columns=data.column_names)
print(data[0]["text"])

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

Instruction:
When did Virgin Australia start operating?

Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [15]:
print(data.column_names)
print(data[0])

['text']
{'text': "Context:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.\n\nInstruction:\nWhen did Virgin Australia start operating?\n\nResponse:\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route."}


In [16]:
import wandb
wandb_api_key = secrets.get_secret("WANDB_SECRET")
status = wandb.login(key=wandb_api_key)
if (status):
    print('Successfully logged into W&B')
else:
    print('Unable to log into W&B')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aadhil2k4 (aadhil2k4-amrita-vishwa-vidyapeetham) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into W&B


In [17]:
wandb.init(project="LlamaTune",)

In [24]:
from trl import SFTTrainer, SFTConfig

# SFT config
sft_config = SFTConfig(
    output_dir="llama3.2-1b-dolly15k-qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    packing=False,
    report_to="wandb",
)

In [25]:
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Verify the conversion
from collections import Counter

print(Counter(
    str(p.dtype)
    for p in model.parameters()
    if p.requires_grad
))

Counter({'torch.float32': 224})


In [26]:
trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    args=sft_config,
)

trainer.train()
trainer.save_model()

Step,Training Loss
10,2.278741
20,2.057700
30,2.135087
40,2.155154
50,2.104418
60,2.047769
70,2.086372
80,2.041996
90,2.017776
100,2.107094


In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "meta-llama/Llama-3.2-1B"
ADAPT = "/kaggle/working/llama3.2-1b-dolly15k-qlora"

for f in ["adapter_config.json", "adapter_model.safetensors"]:
    assert os.path.exists(os.path.join(ADAPT, f)), f"Missing {f} in {ADAPT}"

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(
    BASE, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
model = PeftModel.from_pretrained(model, ADAPT)

prompt = "How does Large Language Models respond to queries?"
text   = f"Instruction:\n{prompt}\n\nResponse:\n"
inputs = tokenizer([text], return_tensors="pt").to(model.device)
out    = model.generate(**inputs, max_new_tokens=200, do_sample=True, top_p=0.9, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:165: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(


Instruction:
How does Large Language Models respond to queries?

Response:
Large Language Models (LLMs) are trained on large amounts of text data and are designed to respond to user queries in natural language. They are capable of understanding and responding to complex questions and queries in a way that is similar to human language. Here are some common ways that LLMs respond to queries:

1. Answering general questions:
LLMs can answer general questions by understanding the context of the question and returning a relevant response. For example, if a user asks "what is the capital of France?", an LLM could respond with the correct answer, "Paris". LLMs can also answer questions that are related to specific topics or concepts, such as "what are the best books on machine learning?".

2. Answering specific questions:
LLMs can also answer specific questions by using context and knowledge to provide a more targeted response. For example, if a user asks "what are the best books on machine l

In [28]:
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16)
peft = PeftModel.from_pretrained(base, ADAPT)
merged = peft.merge_and_unload()    # merges LoRA weights into base
merged.save_pretrained("merged-llama3.2-1b-dolly15k-qlora")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
!tar -cf merged-llama3.2-1b-dolly15k-qlora.tar merged-llama3.2-1b-dolly15k-qlora

In [30]:
print(os.listdir("merged-llama3.2-1b-dolly15k-qlora"))

['model.safetensors', 'generation_config.json', 'config.json']


In [32]:
from huggingface_hub import create_repo, HfApi

# ----- upload -----
repo_id = "aadhil2k4/llama3.2-1b-dolly15k-qlora"   

create_repo(repo_id, private=False)

api = HfApi()
api.upload_folder(
    folder_path="merged-llama3.2-1b-dolly15k-qlora",
    repo_id=repo_id,
    commit_message="Upload merged Llama-3.2-1B Dolly-15k QLoRA model"
)

print(f"Uploaded to https://huggingface.co/{repo_id}")

Uploaded to https://huggingface.co/aadhil2k4/llama3.2-1b-dolly15k-qlora


In [33]:
from transformers import AutoTokenizer
from huggingface_hub import HfApi

repo_id = "aadhil2k4/llama3.2-1b-dolly15k-qlora"

tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
tok.push_to_hub(repo_id)  # uploads tokenizer.json, tokenizer_config.json, etc.

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/aadhil2k4/llama3.2-1b-dolly15k-qlora/commit/717e6c0a0e6f3aa6b91846caeed9772a4f4f97ae', commit_message='Upload tokenizer', commit_description='', oid='717e6c0a0e6f3aa6b91846caeed9772a4f4f97ae', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhil2k4/llama3.2-1b-dolly15k-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhil2k4/llama3.2-1b-dolly15k-qlora'), pr_revision=None, pr_num=None)

In [34]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(


In [35]:
model.push_to_hub("aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter/commit/2fc93b8bcf2a3461032ab59ec27f7523c0a2a99b', commit_message='Upload model', commit_description='', oid='2fc93b8bcf2a3461032ab59ec27f7523c0a2a99b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter'), pr_revision=None, pr_num=None)

In [36]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

model = PeftModel.from_pretrained(base_model, "aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter")

prompt = "Explain the importance of transfer learning in AI."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs_adapter = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs_adapter[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 22.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Explain the importance of transfer learning in AI. Give examples of how transfer learning can be used in different types of AI projects.

Transfer learning is the process of using knowledge from one domain to solve a new problem in another domain. For example, if you have a dataset of images of dogs, you can use this knowledge to train a model to classify images of cats. This process is important for AI because it can help reduce the need for training data from scratch and can make the model more efficient. Transfer learning can also be used for other tasks, such as image captioning and natural language processing.


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

adapter_model = PeftModel.from_pretrained(base_model,"aadhil2k4/llama3.2-1b-dolly15k-qlora-adapter").to(device)

base_model.to(device)

prompt = "Explain the importance of transfer learning in AI."

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs_base = base_model.generate(**inputs, max_new_tokens=150)
decode_base = tokenizer.decode(outputs_base[0], skip_special_tokens=True)

with torch.no_grad():
    outputs_adapter = adapter_model.generate(**inputs, max_new_tokens=150)
decode_adapter = tokenizer.decode(outputs_adapter[0], skip_special_tokens=True)

print("PROMPT:", prompt)
print("BASE MODEL OUTPUT:\n", decode_base)
print("-" * 80)
print("ADAPTER MODEL OUTPUT:\n", decode_adapter)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

PROMPT: Explain the importance of transfer learning in AI.
BASE MODEL OUTPUT:
 Explain the importance of transfer learning in AI. What is transfer learning? Why is it important?

Response:
Transfer learning is a form of artificial intelligence (AI) that uses the knowledge and models of another problem to solve a new problem. Transfer learning is a form of AI that uses the knowledge and models of another problem to solve a new problem. Transfer learning is a form of AI that uses the knowledge and models of another problem to solve a new problem. Transfer learning is a form of AI that uses the knowledge and models of another problem to solve a new problem. Transfer learning is a form of AI that uses the knowledge and models of another problem to solve a new problem. Transfer learning is a form of AI that uses the knowledge and models of another problem to solve a new problem. Transfer learning
--------------------------------------------------------------------------------
ADAPTER MODEL 